# IBM HR Attrition Analysis

I'm starting by loading the raw employee dataset. This should contain one row per employee with various demographic and job-related features.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

raw = pd.read_csv("../raw_unedited/WA_Fn-UseC_-HR-Employee-Attrition.csv")
raw.head()

Next, I'm checking for duplicate rows and trimming text columns before anything else touches this data. I'm also running a quick sanity check to ensure `EmployeeNumber` is actually unique.

In [ ]:
d = raw.copy()
for c in d.select_dtypes(include=["object", "string"]).columns:
    d[c] = d[c].astype("string").str.strip()
    d[c] = d[c].replace({"": pd.NA})

assert d["EmployeeNumber"].is_unique, "EmployeeNumber is not unique!"
print(f"Total unique employees: {len(d)}")

I need to validate the logic of the tenure columns. It wouldn't make sense for someone to be in their current role longer than they've been at the company, or at the company longer than their total working years.

In [ ]:
tenure_errors = (d["YearsInCurrentRole"] > d["YearsAtCompany"]).any() or \
                (d["YearsAtCompany"] > d["TotalWorkingYears"]).any()
assert not tenure_errors, "Found logical contradictions in employee tenure data!"
print("Tenure logic validated successfully.")

Finally, I want to calculate the attrition rate by job role to identify which positions are experiencing the highest turnover.

In [ ]:
rate = d.groupby("JobRole", observed=True).agg(
    n=("EmployeeNumber", "size"),
    attritions=("Attrition", lambda x: (x == "Yes").sum())
).reset_index()
rate["attrition_rate"] = rate.attritions / rate.n

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=rate.sort_values("attrition_rate", ascending=False), y="JobRole", x="attrition_rate", color="#327a8b", ax=ax)
ax.set(title="Attrition rate by job role", xlabel="Attrition share", ylabel="Job role")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
fig.tight_layout()
plt.show()